Structured output

Models can be requested to provide their response in format matching in a given schema. This is useful for ensuring the output can be easily parsed  and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output. 



Pydantic

Pydantic models that provide the richest feature set with field validation , description and nested structure.

In [6]:
import os
from langchain.chat_models import init_chat_model
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1")
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002367DCA96D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002367DCA9BD0>, root_client=<openai.OpenAI object at 0x000002367DCA9450>, r

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int= Field(description="This year the movie was released.")
    director:str= Field(description="This is the director of the movie.")
    rating:float = Field(description="The movies rating out of 10.")

In [8]:
model_with_structure=model.with_structured_output(Movie)
response =model_with_structure.invoke("Provide details about the movie?")

### Message output alongside parse structure

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(..., description="The title of the movie")
    year:int= Field(...,description="This year the movie was released.")
    director:str= Field(...,description="This is the director of the movie.")
    rating:float = Field(...,description="The movies rating out of 10.")

In [10]:
model_with_structure=model.with_structured_output(Movie, include_raw=True)
response =model_with_structure.invoke("Provide details about the Bang bang movie?")
response

{'raw': AIMessage(content='{"title":"Bang Bang!","year":2014,"director":"Siddharth Anand","rating":5.5}', additional_kwargs={'parsed': Movie(title='Bang Bang!', year=2014, director='Siddharth Anand', rating=5.5), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 128, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_04d28f6feb', 'id': 'chatcmpl-EZEIswUdRij6eGfntVd6StsCxNZxX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13f68-66be-7291-a45d-6c76d242ef75-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 128, 'output_tokens': 2

### Nested Structure

In [14]:
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    """A movie with details"""
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    director:str
    budget:float | None= Field(None, description="Budget in million INR") 
    profit:float | None = Field(None, description="Profit earned in million INR")


model_with_structure=model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about the Yeh Jawani hai diwani.")
response

{'title': 'Yeh Jawaani Hai Deewani',
 'year': 2013,
 'cast': [{'name': 'Ranbir Kapoor', 'role': "Kabir 'Bunny' Thapar"},
  {'name': 'Deepika Padukone', 'role': 'Naina Talwar'},
  {'name': 'Aditya Roy Kapur', 'role': "Avinash 'Avi' Arora"},
  {'name': 'Kalki Koechlin', 'role': 'Aditi Mehra'}],
 'genres': ['Romance', 'Drama', 'Comedy'],
 'director': 'Ayan Mukerji',
 'budget': 50,
 'profit': 196}

TypeDict
Typedict provides a simpler alternative using python's built-in typing , ideal when we dont need run time validation.

In [13]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with Details."""
    title: Annotated[str, ... , "This is title of the movie."]
    year: Annotated[int,...,"The year movie was released."]
    director: Annotated[str, ... , "Director of the movie."]
    rating: Annotated[float, ... , "Rating of the movie out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide the details about the Garam Hawa.")
response



{'title': 'Garam Hawa', 'year': 1973, 'director': 'M.S. Sathyu', 'rating': 8.2}

In [16]:
model.profile

{'name': 'GPT-4.1',
 'release_date': '2025-04-14',
 'last_updated': '2025-04-14',
 'open_weights': False,
 'max_input_tokens': 1047576,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'pdf_inputs': True,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True}

DataClass

Dataclass is mainly class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str= Field(description="This is the name of the person.")
    email:str= Field(description="This is the email address of the person.")
    phone:str= Field(description="This is the phone number of the person.")

agent = create_agent(
    model="gpt-4.1",
    response_format=ContactInfo
   ) 

result = agent.invoke({
    "messages" :[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [27]:
##TYPEDICT  

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for some person"""
    name:str
    email:str
    phone:str


agent = create_agent(
    model="gpt-4.1",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{
        "role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [28]:
#Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for some person"""
    name:str
    email:str
    phone:str

agent = create_agent(
    model="gpt-4.1",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{
        "role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"
    }]
})    

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')